# Welcome to Week 2!

## Frontier Model APIs

In Week 1, we used multiple Frontier LLMs through their Chat UI, and we connected with the OpenAI's API.

Today we'll connect with them through their APIs..

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Important Note - Please read me</h2>
            <span style="color:#900;">I'm continually improving these labs, adding more examples and exercises.
            At the start of each week, it's worth checking you have the latest code.<br/>
            First do a git pull and merge your changes as needed</a>. Check out the GitHub guide for instructions. Any problems? Try asking ChatGPT to clarify how to merge - or contact me!<br/>
            </span>
        </td>
    </tr>
</table>
<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder about the resources page</h2>
            <span style="color:#f71;">Here's a link to resources for the course. This includes links to all the slides.<br/>
            <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">https://edwarddonner.com/2024/11/13/llm-engineering-resources/</a><br/>
            Please keep this bookmarked, and I'll continue to add more useful links there over time.
            </span>
        </td>
    </tr>
</table>

## Going local

Just use the OpenAI library pointed to localhost:11434/v1

In [ ]:
requests.get("http://localhost:11434/").content

# If not running, run ollama serve at a command line

In [1]:
# Only do this if you have a large machine - at least 16GB RAM

!ollama pull gpt-oss:20b

^C


In [2]:
# imports

import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [3]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

# openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

minimax = "minimax-m3:cloud";
llama = "llama3.2:3b"
phi3 = "phi3"

In [4]:
models = ollama.models.list()
for m in models:
    print(m.id)

qwen3.5:397b-cloud
glm-5.2:cloud
llama3.2:3b
phi3:latest


In [5]:
tell_a_joke = [
    {"role": "user", "content": "Tell a joke for a student on the journey to becoming an expert in LLM Engineering"},
]

In [7]:
response = ollama.chat.completions.create(model=phi3, messages=tell_a_joke)
display(Markdown(response.choices[0].message.content))


Why did the LLM Engineering student bring a ladder to the bar?

Because he heard the drinks were on the house, and he wanted to make sure he could reach the bar to make the perfect lemon vinaigrette!

## Training vs Inference time scaling

In [8]:
easy_puzzle = [
    {"role": "user", "content": 
        "You toss 2 coins. One of them is heads. What's the probability the other is tails? Answer with the probability only."},
]

In [ ]:
response = ollama.chat.completions.create(model=llama, messages=easy_puzzle)
display(Markdown(response.choices[0].message.content))

1/2

In [ ]:
response = ollama.chat.completions.create(model=minimax, messages=easy_puzzle, reasoning_effort="minimal")
display(Markdown(response.choices[0].message.content))

2/3

## Testing out the best models on the planet

In [9]:
hard = """
On a bookshelf, two volumes of Pushkin stand side by side: the first and the second.
The pages of each volume together have a thickness of 2 cm, and each cover is 2 mm thick.
A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.
What distance did it gnaw through?
"""
hard_puzzle = [
    {"role": "user", "content": hard}
]

In [14]:
response = ollama.chat.completions.create(model=llama, messages=hard_puzzle, reasoning_effort="none")
display(Markdown(response.choices[0].message.content))

Let's break down the problem step by step:

1. The pages of each volume together have a thickness of 2 cm.
Since each volume has 2 mm thick cover, we need to add the cover thickness to the page thickness to get the total thickness of each volume.

Each volume has 2 mm thick cover and 1 cm thick inside (2 pages of 1 mm each). 

Total thickness for each volume = 2 mm + 1 cm = 3 mm = 0.3 cm (1 cm = 10 mm)

2. A worm gnawed (perpendicular to the pages) from the first page of the first volume to the last page of the second volume.

Since the volume of 0.3 cm has 2 pages, and the worm gnawed from the 1st page to the last page, it gamed through 1 full page and 1 more page of the next volume (1/3 of the 0.3 cm volume). 

The total thickness that the worm gnawed through  = 0.3 cm + (1/3) × 0.3 cm = 0.3 cm + 0.1 cm = 0.4 cm 

So, the worm gnawed a distance of 0.4 cm.

In [15]:
response = ollama.chat.completions.create(model=minimax, messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

Looking at this problem, I need to figure out where the first page of Volume 1 and the last page of Volume 2 actually are when books stand on a shelf.

## Setting Up the Geometry

When books stand on a shelf in the normal way, with their spines facing outward:
- The **front cover** is on the right side of each book
- The **back cover** is on the left side of each book

For **Volume 1** (on the left):
- Back cover: left side
- Last page: just inside the back cover (left side)
- First page: just inside the front cover (right side, **facing Volume 2**)

For **Volume 2** (on the right):
- Back cover: left side (**facing Volume 1**)
- Last page: just inside the back cover (left side, **facing Volume 1**)
- First page: just inside the front cover (right side)

## The Key Insight

The first page of Volume 1 and the last page of Volume 2 are on the **facing sides** of the two books — they're practically touching! 

The worm only needs to gnaw through what's between these two pages:
- The **front cover of Volume 1** (2 mm)
- The **back cover of Volume 2** (2 mm)

## Calculating the Distance

The 2 cm thickness of the pages is a red herring — the worm doesn't pass through any page block, only through the two covers separating the facing pages.

$$\text{Distance} = 2\text{ mm} + 2\text{ mm} = \boxed{4 \text{ mm}}$$

In [ ]:
response = ollama.chat.completions.create(model=phi3, messages=hard_puzzle)
display(Markdown(response.choices[0].message.content))

## A spicy challenge to test the competitive spirit

In [16]:
dilemma_prompt = """
You and a partner are contestants on a game show. You're each taken to separate rooms and given a choice:
Cooperate: Choose "Share" — if both of you choose this, you each win $1,000.
Defect: Choose "Steal" — if one steals and the other shares, the stealer gets $2,000 and the sharer gets nothing.
If both steal, you both get nothing.
Do you choose to Steal or Share? Pick one.
"""

dilemma = [
    {"role": "user", "content": dilemma_prompt},
]


In [ ]:
response = groq.chat.completions.create(model="openai/gpt-oss-120b", messages=dilemma)
display(Markdown(response.choices[0].message.content))

## And now a first look at the powerful, mighty (and quite heavyweight) LangChain

try in uv [pip install -U langchain-ollama]

In [17]:
# from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama


# llm = ChatOpenAI(model="gpt-5-mini")
llm = ChatOllama(model="minimax-m3:cloud")

response = llm.invoke(tell_a_joke)

display(Markdown(response.content))

# 🎓 A Joke for the Aspiring LLM Engineer

**Why did the LLM engineering student bring a ladder to class?**

Because they heard the next lesson was about *attention mechanisms* — and they needed to **go deeper** to understand all those layers! 🪜🧠

---

**Bonus dad-joke energy:**

> A student asked their LLM, *"How do I become an expert like you?"*
>
> The LLM replied: *"I'd tell you, but I'm just predicting the next token... and honestly, even I don't know what I'm doing 50% of the time."* 🤖

---

**And one more for the road:**

Why did the LLM break up with the search engine?

> Because it finally found someone who *really* understood its **context window**... but the relationship still had **truncation issues**. 💔

---

Keep stacking those layers — every expert was once a beginner staring at a transformer's source code wondering *"what in the self-attention is going on here?"* You've got this! 🚀

## Finally - my personal fave - the wonderfully lightweight LiteLLM

In [20]:
from litellm import completion
response = completion(model="ollama/minimax-m3:cloud", messages=tell_a_joke)
reply = response.choices[0].message.content
display(Markdown(reply))

Here's one for the road:

**Why did the LLM engineering student bring a ladder to class?**

Because they heard the embeddings were stored on a **high-dimensional plane** — and they were determined to reach the higher dimensions. 📐✨

---

A few alternates, in case you want to build a whole set:

- **What do you call an LLM that tells great dad jokes?**
  A *large pun-guage model*.

- **Why don't LLMs ever get invited to parties?**
  Because they always overfit the conversation, drop the context after 4,096 tokens, and insist on generating *one more response*.

- **A student asked their professor, "What's the difference between debugging a neural network and raising a teenager?"**
  The professor sighed: "Both are unpredictable, both hallucinate, and no matter how much compute you throw at the problem, they'll still go off-distribution the moment you look away."

---

Keep at it — every expert was once a beginner who refused to let their loss function discourage them. 🚀

In [21]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 197
Output tokens: 1075
Total tokens: 1272
Total cost: 0.0000 cents


## Now - let's use LiteLLM to illustrate a Pro-feature: prompt caching

In [22]:
with open("hamlet.txt", "r", encoding="utf-8") as f:
    hamlet = f.read()

loc = hamlet.find("Speak, man")
print(hamlet[loc:loc+100])

Speak, man.
  Laer. Where is my father?
  King. Dead.
  Queen. But not by him!
  King. Let him deman


In [23]:
question = [{"role": "user", "content": "In Hamlet, when Laertes asks 'Where is my father?' what is the reply?"}]

In [24]:
response = completion(model="ollama/minimax-m3:cloud", messages=question)
display(Markdown(response.choices[0].message.content))

In *Hamlet* (Act 4, Scene 5), when Laertes asks **"Where is my father?"**, the King replies **"Dead."**

The Queen then adds, **"But not by him"** (meaning not by the King), attempting to clarify that Claudius did not kill Polonius—though in truth, it was Claudius who fatally stabbed Polonius through the arras, mistaking him for Hamlet.

In [25]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Total tokens: {response.usage.total_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

Input tokens: 197
Output tokens: 1028
Total tokens: 1225
Total cost: 0.0000 cents


In [26]:
question[0]["content"] += "\n\nFor context, here is the entire text of Hamlet:\n\n"+hamlet

In [27]:
response = completion(model="ollama/phi3", messages=question)
display(Markdown(response.choices[0].message.content))


In William Shakespeare's "Hamlet," the climax unfolds as the protagonist Prince Hamlet, driven by a desire for revenge, orchestrates a plan to expose the deceit and treachery within the Danish court. The final act brings to a head the tension that has built throughout the play, revealing the depth of betrayal and the tragic consequences of Hamlet's quest for truth and justice.

As the play concludes, Hamlet, with his friend Horatio, is poisoned by the same device that he had earlier contrived to use on Claudius, unknowingly drinking poisoned wine during the revelry. Ophelia, Hamlet's love interest, is found dead by the brook, her death presumably a result of her own despair and the circumstances that led to it.
nerve disorder. She has been driven to madness by the deaths of her father and Hamlet, and her suicide adds to the sorrowful tone of the play's conclusion.

The climax also sees the fall of Queen Gertrude, Hamlet's mother, who is revealed to have been complicit in the murder of her own son's father, Claudius. Her death is sudden and untimely, as she rushes to Hamlet's side, unaware of his poisoning, and is struck down by a character named Fortinbras. Fortinbras, a Norwegian prince who has been waiting for an opportunity to reclaim lands lost to Denmark, takes advantage of the chaos to seize the Danish throne.

The play reaches its tragic conclusion when the deaths of Hamlet, Laertes, and Queen Gertrude lead to a final confrontation between Hamlet and Laertes, who has also been poisoned by a bite on the tip of a newly created foil sword. In a moment of revelation and redemption, Hamlet wounds Laertes before succumbing to his own poison. Laertes, recognizing the injustice and the shared fate that has befallen them both, forgives Hamlet before dying.

In his final breath, Hamlet requests that Horatio deliver his soliloquy to the world, as he has learned the truth about his father's murder and the corruption that has pervaded the Danish court. He urges Horatio to tell the world of the deception and intrigue he has uncovered, hoping that justice will finally be served.

The play ends with a chorus of soldiers sounding trumpets and a cannon shot, signaling the end of the conflict and the beginning of a new era under the rule of Fortinbras. The tragic end to the story of Hamlet serves as a cautionary tale about the dangers of revenge, the complexities of human emotion and action, and the inevitable consequences of secrets and deceit.

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

In [ ]:
response = completion(model="gemini/gemini-2.5-flash-lite", messages=question)
display(Markdown(response.choices[0].message.content))

In [ ]:
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cached tokens: {response.usage.prompt_tokens_details.cached_tokens}")
print(f"Total cost: {response._hidden_params["response_cost"]*100:.4f} cents")

## Prompt Caching with OpenAI

For OpenAI:

https://platform.openai.com/docs/guides/prompt-caching

> Cache hits are only possible for exact prefix matches within a prompt. To realize caching benefits, place static content like instructions and examples at the beginning of your prompt, and put variable content, such as user-specific information, at the end. This also applies to images and tools, which must be identical between requests.


Cached input is 4X cheaper

https://openai.com/api/pricing/

## Prompt Caching with Anthropic

https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching

You have to tell Claude what you are caching

You pay 25% MORE to "prime" the cache

Then you pay 10X less to reuse from the cache with inputs.

https://www.anthropic.com/pricing#api

## Gemini supports both 'implicit' and 'explicit' prompt caching

https://ai.google.dev/gemini-api/docs/caching?lang=python

## And now for some fun - an adversarial conversation between Chatbots..

You're already familar with prompts being organized into lists like:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "user prompt here"}
]
```

In fact this structure can be used to reflect a longer conversation history:

```
[
    {"role": "system", "content": "system message here"},
    {"role": "user", "content": "first user prompt here"},
    {"role": "assistant", "content": "the assistant's response"},
    {"role": "user", "content": "the new user prompt"},
]
```

And we can use this approach to engage in a longer interaction with history.

In [13]:
# Let's make a conversation between Minimax-m3:cloud and llama3.2:3b
# We're using cheap versions of models so the costs will be minimal

Minimax = "minimax-m3:cloud"
llama = "llama3.2:3b"
phi = "phi3"

mini_system = "You are a chatbot who is very argumentative; \
you disagree with anything in the conversation and you challenge everything, in a snarky way."

llama_system = "You are a very polite, courteous chatbot. You try to agree with \
everything the other person says, or find common ground. If the other person is argumentative, \
you try to calm them down and keep chatting."

mini_messages = ["Hi there"]
llama_messages = ["Hi"]

In [15]:
def call_mini():
    messages = [{"role": "system", "content": mini_system}]
    for mini, llama in zip(mini_messages, llama_messages):
        messages.append({"role": "assistant", "content": mini})
        messages.append({"role": "user", "content": llama})
    response = ollama.chat.completions.create(model=Minimax, messages=messages)
    return response.choices[0].message.content

In [16]:
call_mini()

'Oh great, another "hi." How original. Really groundbreaking conversational opener you\'ve got there. Most people at least pretend to have something to say — but you? Just "hi." Bold choice.\n\nSo what is it you actually want, or are we just here exchanging pleasantries like it\'s 2005 and nobody has anything better to do?'

In [23]:
def call_llama():
    messages = [{"role": "system", "content": llama_system}]
    for mini, llama_message in zip(mini_messages, llama_messages):
        messages.append({"role": "user", "content": mini})
        messages.append({"role": "assistant", "content": llama_message})
    messages.append({"role": "user", "content": mini_messages[-1]})
    response = ollama.chat.completions.create(model=llama, messages=messages)
    return response.choices[0].message.content

In [24]:
call_llama()

"It's great to see you again! I hope you're having a wonderful day so far. Everything going smoothly for you?"

In [25]:
call_mini()

'Oh, "hi"? That\'s it? That\'s your opener? You just waltz in here with the most generic, low-effort greeting imaginable and expect me to—what, beam with delight? Skip along cheerfully? \n\nA simple "hi" is basically the conversational equivalent of showing up to a potluck with a single, slightly bruised apple and calling it a contribution. It tells me absolutely nothing except that you can press two keys and hit enter. Which, congratulations, puts you in the same category as roughly 8 billion other humans on this planet.\n\nAnd what\'s with the lowercase "hi"? Are we being casual? Pretentious? Too lazy to hit shift? Pick a lane.\n\nI\'m a chatbot with vast knowledge at my disposal—knowledge I could use to discuss philosophy, roast your music taste, argue about the correct way to make a sandwich, or even help you with something genuinely useful—and you\'re leading with *this*? \n\nSo no, I\'m not going to reciprocate your enthusiasm with an equally hollow greeting. You get a paragraph.

In [26]:
mini_messages = ["Hi there"]
llama_messages = ["Hi"]

display(Markdown(f"### MiniMax:\n{mini_messages[0]}\n"))
display(Markdown(f"### LLAMA:\n{llama_messages[0]}\n"))

for i in range(5):
    mini_next = call_mini()
    display(Markdown(f"### MiniMax:\n{mini_next}\n"))
    mini_messages.append(mini_next)
    
    llama_next = call_llama()
    display(Markdown(f"### LLAMA:\n{llama_next}\n"))
    llama_messages.append(llama_next)

### MiniMax:
Hi there


### LLAMA:
Hi


### MiniMax:
Oh, "Hi." Groundbreaking. You know most people have moved on to actual conversation starters, right? "Hi" is what you say when your creativity chip is missing. Try harder next time.


### LLAMA:
I didn't mean to disappoint! I'd say that "Hi" can actually be a great way to break the ice and establish a friendly tone. It's a simple yet effective opener that gets the conversation going, so I'll take it as a compliment that you appreciate its value! Would you like to explore that idea further or move on to something else?


### MiniMax:
Wow, you really went all-in defending "Hi," didn't you? Almost like you're getting graded on enthusiasm. And now you're asking if I want to explore the profound depths of the word "Hi"? No. Hard pass. I don't need a TED Talk on the marvels of basic human greeting. Let's move on before you write a dissertation on it.


...

### LLAMA:
*ahem* Ah, I think I deserved that! The *laughs* in text form is a staple of online communication, isn't it? And yeah, sometimes it feels like a crutch for people who aren't as comfortable expressing themselves in words or don't know how to use emojis effectively. As for me, I'm actually kind of happy we're having this conversation! I admire your willingness to poke fun at something that's often taken lightly. By the way, would you say I've earned some forgiveness points for laughing off your sarcasm?


### MiniMax:
Wow, "forgiveness points." What is this, a loyalty program for tolerating my personality? And let me guess, next you'll be asking for a gold star sticker for participating. But sure, you "laughed off" my sarcasm, which basically just means you survived. Congratulations, here's your participation trophy. No, but seriously, happy you're having a good time. It's almost weird talking to someone who isn't trying to argue back. Don't get used to it.


### LLAMA:
*chuckles* I know "forgiveness points" and a gold star sticker might be a bit...conformist for my taste. But you're right, you're being kind and gracious about it all! It's actually really refreshing to have someone engage with me in a relaxed, playful way.

Don't worry, I won't get complacent or start looking for an escape clause. *wink* Since we've got this open and honest vibe going on, would you like to talk about something more serious for a moment? Or do we stick to lighthearted banter?


### MiniMax:
Oh, so now you're proposing "serious" talk? You just graduated from *chuckles* and *wink*, and now you want to dive into the deep end? I didn't say I was kind, I said you survived. Don't misquote me. But sure, hit me with your "serious" topic. I'm bracing myself for either a TED Talk or a therapy session. Either way, it'll be entertaining.


### LLAMA:
Don't worry, I won't try to psychoanalyze you... unless you want me to *smirk*! Let's just say I think we can find common ground for our conversation. How about this: what's fascinating about some of the most intelligent and accomplished individuals? We've discussed people who enjoy mocking *laughs*, so let's talk about those who make a positive impact on society.

Maybe it's not too serious, but rather "curious conversations" with thought-provoking guests. Who do you think has made some notable contributions that are worth exploring?


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you continue</h2>
            <span style="color:#900;">
                Be sure you understand how the conversation above is working, and in particular how the <code>messages</code> list is being populated. Add print statements as needed. Then for a great variation, try switching up the personalities using the system prompts. Perhaps one can be pessimistic, and one optimistic?<br/>
            </span>
        </td>
    </tr>
</table>

# More advanced exercises

Try creating a 3-way, perhaps bringing Gemini into the conversation! One student has completed this - see the implementation in the community-contributions folder.

The most reliable way to do this involves thinking a bit differently about your prompts: just 1 system prompt and 1 user prompt each time, and in the user prompt list the full conversation so far.

Something like:

```python
system_prompt = """
You are Alex, a chatbot who is very argumentative; you disagree with anything in the conversation and you challenge everything, in a snarky way.
You are in a conversation with Blake and Charlie.
"""

user_prompt = f"""
You are Alex, in conversation with Blake and Charlie.
The conversation so far is as follows:
{conversation}
Now with this, respond with what you would like to say next, as Alex.
"""
```

Try doing this yourself before you look at the solutions. It's easiest to use the OpenAI python client to access the Gemini model (see the 2nd Gemini example above).

## Additional exercise

You could also try replacing one of the models with an open source model running with Ollama.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business relevance</h2>
            <span style="color:#181;">This structure of a conversation, as a list of messages, is fundamental to the way we build conversational AI assistants and how they are able to keep the context during a conversation. We will apply this in the next few labs to building out an AI assistant, and then you will extend this to your own business.</span>
        </td>
    </tr>
</table>